In [1]:
"""
SIH26153 — Coder B: The World Model Core
==========================================
This script implements the "world model" core described in the task brief:
  1. Load Coder A's normalized feature matrix   (STEP 1 - currently a DUMMY generator)
  2. Structure data into fixed-length time windows            (STEP 2)
  3. Build an LSTM in PyTorch: input = windowed sequence,
     output = predicted next-state feature vector + infiltration probability (STEP 3)
  4. Train it to predict P(state_{t+1} | state_t)              (STEP 4)
  5. K-step forward simulation -> probability-of-infiltration curve (STEP 5)
  6. Compare F1 / precision / recall / FPR against a logistic regression baseline (STEP 6)
  7. Save trained model weights + training config               (STEP 7)

HOW TO SWAP IN REAL DATA (once Coder A delivers):
  - Replace the `load_feature_matrix()` function body with Coder A's actual
    loading logic / imported module. It must return:
        features: np.ndarray of shape (N, F)   -> N flows, F normalized features
        labels:   np.ndarray of shape (N,)      -> 0 = benign, 1 = malicious, per flow
  - Everything downstream (windowing, model, training, evaluation) stays the same.
"""

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
    roc_auc_score, precision_recall_curve,
)
import matplotlib
matplotlib.use("Agg")  # no display needed, just save to file
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# CONFIG (kept in one place so it's easy to log / save for reproducibility)
# ----------------------------------------------------------------------------
CONFIG = {
    "window_size": 10,      # STEP 2: TL's recommendation - 10-flow windows
    "k_steps": 5,            # STEP 5: K-step forward simulation, K=5 to start
    "hidden_size": 64,
    "num_layers": 2,
    "batch_size": 32,
    "epochs": 15,
    "early_stopping_patience": 3,  # stop if val_loss doesn't improve for this many epochs
    "learning_rate": 1e-3,
    "infiltration_loss_weight": 1.0,   # weight for BCE loss (infiltration prob)
    "state_loss_weight": 1.0,          # weight for MSE loss (next-state prediction)
    "pos_class_weight": None,          # set at runtime from actual positive rate (Fix #1/#3)
    "seed": 42,
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ----------------------------------------------------------------------------
# STEP 1: Load feature matrix (DUMMY VERSION — swap this out for Coder A's data)
# ----------------------------------------------------------------------------
def load_feature_matrix(n_flows: int = 5000, n_features: int = 16):
    """
    DUMMY DATA GENERATOR.
    Simulates what Coder A's normalized feature matrix + labels should look like.

    Returns
    -------
    features : np.ndarray, shape (N, F)   - already "normalized" (mean~0, std~1)
    labels   : np.ndarray, shape (N,)     - 0 = benign, 1 = malicious flow
    """
    rng = np.random.default_rng(CONFIG["seed"])

    # Simulate a slowly-evolving "attack campaign": a rising baseline hazard
    # that occasionally spikes, so there IS real temporal structure to learn
    # (a real classifier-only model would miss this progression).
    hazard = np.linspace(0.02, 0.15, n_flows) + 0.05 * np.sin(np.linspace(0, 20, n_flows))
    hazard = np.clip(hazard, 0.01, 0.9)
    labels = (rng.random(n_flows) < hazard).astype(np.float32)

    # Features correlate weakly with the hazard/label + noise, like real
    # network features (packet size, TTL variance, inter-arrival time, etc.)
    base = rng.normal(0, 1, size=(n_flows, n_features))
    signal = labels.reshape(-1, 1) * rng.normal(2.0, 0.5, size=(1, n_features))
    features = base + signal
    features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)

    return features.astype(np.float32), labels.astype(np.float32)


# ----------------------------------------------------------------------------
# STEP 2: Build fixed-length time windows
# ----------------------------------------------------------------------------
def make_windows(features: np.ndarray, labels: np.ndarray, window_size: int):
    """
    Slides a window of `window_size` flows across the data.

    For each window ending at index i (i.e. flows [i-window_size+1 ... i]):
      X          = the window itself                -> (window_size, F)
      next_state = the feature vector AFTER the window (flow i+1)  -> (F,)
      next_label = whether the NEXT flow is malicious -> scalar
                   (this is the ground truth for "infiltration probability
                    at t+1", i.e. P(state_{t+1} | state_t))

    Returns
    -------
    X          : (num_windows, window_size, F)
    next_state : (num_windows, F)
    next_label : (num_windows,)
    """
    N, F = features.shape
    X, next_state, next_label = [], [], []

    for i in range(window_size, N - 1):
        window = features[i - window_size:i]      # (window_size, F)
        X.append(window)
        next_state.append(features[i])             # the very next flow's features
        next_label.append(labels[i])                # infiltration ground truth at t+1

    return (
        np.array(X, dtype=np.float32),
        np.array(next_state, dtype=np.float32),
        np.array(next_label, dtype=np.float32),
    )


class WindowDataset(Dataset):
    """Simple PyTorch Dataset wrapper around the windowed arrays."""

    def __init__(self, X, next_state, next_label):
        self.X = torch.from_numpy(X)
        self.next_state = torch.from_numpy(next_state)
        self.next_label = torch.from_numpy(next_label)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.next_state[idx], self.next_label[idx]


# ----------------------------------------------------------------------------
# STEP 3: The World Model (LSTM) — predicts next-state vector + infiltration prob
# ----------------------------------------------------------------------------
class WorldModel(nn.Module):
    """
    Input : (batch, window_size, n_features)  -- a windowed sequence of flows
    Output:
        next_state_pred      : (batch, n_features)  -- predicted next feature vector
        infiltration_prob    : (batch,)              -- P(malicious at t+1 | state_t)

    This is NOT a plain classifier: the state-prediction head is what makes it
    a "world model" (it forecasts the next feature vector), and the
    infiltration head is trained on top of the same learned representation.
    """

    def __init__(self, n_features: int, hidden_size: int, num_layers: int):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.state_head = nn.Linear(hidden_size, n_features)   # predict next state
        self.infiltration_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (batch, window_size, n_features)
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]              # (batch, hidden_size) - final timestep
        next_state_pred = self.state_head(last_hidden)             # (batch, n_features)
        infiltration_prob = self.infiltration_head(last_hidden).squeeze(-1)  # (batch,)
        return next_state_pred, infiltration_prob, last_hidden


# ----------------------------------------------------------------------------
# STEP 4: Training loop -- P(state_{t+1} | state_t)
# ----------------------------------------------------------------------------
def train_model(model, train_loader, val_loader, config):
    """
    Trains with EARLY STOPPING: the epoch with the BEST validation AUC-ROC is
    kept (not just lowest val_loss -- with class imbalance, val_loss can look
    fine while the model still ignores the rare positive class entirely).

    FIX #3 (class weighting): uses BCEWithLogitsLoss with pos_weight so that
    missing a real infiltration (false negative) is penalized harder than a
    false positive, proportional to how rare positives actually are.

    FIX #2 (AUC-ROC): computed every epoch on validation data. AUC is
    threshold-independent, so it tells us whether the model has learned any
    real signal at all, separate from whether 0.5 is the right cutoff.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    mse_loss = nn.MSELoss()
    # pos_weight up-weights the rare "infiltration=1" class in the loss.
    # infiltration_head already ends in Sigmoid, so we use BCELoss with manual
    # weighting here (mirrors BCEWithLogitsLoss(pos_weight=...) behaviour).
    pos_weight = config["pos_class_weight"]

    def weighted_bce(pred, target):
        eps = 1e-7
        pred = pred.clamp(eps, 1 - eps)
        weights = torch.where(target == 1, pos_weight, torch.tensor(1.0, device=pred.device))
        loss = -weights * (target * torch.log(pred) + (1 - target) * torch.log(1 - pred))
        return loss.mean()

    best_val_auc = -1.0
    best_state_dict = None
    best_epoch = 0
    epochs_since_improvement = 0

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        total_loss = 0.0
        for X, next_state, next_label in train_loader:
            X, next_state, next_label = X.to(DEVICE), next_state.to(DEVICE), next_label.to(DEVICE)

            optimizer.zero_grad()
            pred_state, pred_infil, _ = model(X)

            loss_state = mse_loss(pred_state, next_state)
            loss_infil = weighted_bce(pred_infil, next_label)
            loss = (
                config["state_loss_weight"] * loss_state
                + config["infiltration_loss_weight"] * loss_infil
            )

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X.size(0)

        avg_train_loss = total_loss / len(train_loader.dataset)

        # validation pass -- collect probs/labels to compute AUC-ROC
        model.eval()
        val_loss = 0.0
        all_probs, all_labels = [], []
        with torch.no_grad():
            for X, next_state, next_label in val_loader:
                X, next_state, next_label = X.to(DEVICE), next_state.to(DEVICE), next_label.to(DEVICE)
                pred_state, pred_infil, _ = model(X)
                l = mse_loss(pred_state, next_state) + weighted_bce(pred_infil, next_label)
                val_loss += l.item() * X.size(0)
                all_probs.append(pred_infil.cpu().numpy())
                all_labels.append(next_label.cpu().numpy())
        avg_val_loss = val_loss / len(val_loader.dataset)
        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)

        # AUC needs both classes present in the validation batch to be defined
        if len(np.unique(all_labels)) > 1:
            val_auc = roc_auc_score(all_labels, all_probs)
        else:
            val_auc = float("nan")

        improved = val_auc > best_val_auc
        marker = "  <- best so far (by AUC)" if improved else ""
        print(f"Epoch {epoch:2d}/{config['epochs']}  "
              f"train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}  "
              f"val_auc={val_auc:.4f}{marker}")

        if improved:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_since_improvement = 0
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= config["early_stopping_patience"]:
                print(f"Early stopping: no AUC improvement for "
                      f"{config['early_stopping_patience']} epochs.")
                break

    # Restore the BEST epoch's weights (highest val AUC, not just lowest loss)
    model.load_state_dict(best_state_dict)
    print(f"Restored weights from best epoch: {best_epoch} (val_auc={best_val_auc:.4f})\n")

    return model


# ----------------------------------------------------------------------------
# STEP 5: K-step forward simulation -> probability-of-infiltration curve
# ----------------------------------------------------------------------------
@torch.no_grad()
def k_step_rollout(model, initial_window: np.ndarray, k: int):
    """
    Given a single starting window (window_size, F), roll the model forward
    K times, feeding each predicted next-state back in as the newest step
    (dropping the oldest step), and collect the infiltration probability at
    each of the K steps.

    Returns
    -------
    probs : list[float] of length k -- the probability-of-infiltration curve
    """
    model.eval()
    window = torch.from_numpy(initial_window).unsqueeze(0).to(DEVICE)  # (1, window_size, F)
    probs = []

    for step in range(k):
        pred_state, pred_infil, _ = model(window)
        probs.append(pred_infil.item())

        # slide the window: drop oldest timestep, append predicted next state
        pred_state_expanded = pred_state.unsqueeze(1)          # (1, 1, F)
        window = torch.cat([window[:, 1:, :], pred_state_expanded], dim=1)

    return probs


def plot_rollout_curve(probs, out_path="infiltration_curve.png"):
    """
    Saves a simple line chart of the K-step probability-of-infiltration curve.
    This is the single most 'showable' artifact from Coder B's work --
    drop this PNG straight into the 5-slide tech presentation or screen-record
    it for the 2-minute demo video, even with no frontend/UI built.
    """
    steps = list(range(1, len(probs) + 1))
    plt.figure(figsize=(6, 4))
    plt.plot(steps, probs, marker="o", linewidth=2)
    plt.axhline(0.5, color="red", linestyle="--", linewidth=1, label="0.5 decision threshold")
    plt.xlabel("Steps ahead (t+k)")
    plt.ylabel("P(infiltration)")
    plt.title("World Model: K-Step Infiltration Probability Forecast")
    plt.ylim(0, 1)
    plt.xticks(steps)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved rollout chart -> {out_path}")


# ----------------------------------------------------------------------------
# STEP 6: Baseline comparison (logistic regression) + shared metrics
# ----------------------------------------------------------------------------
def compute_metrics(y_true, y_pred_binary):
    f1 = f1_score(y_true, y_pred_binary, zero_division=0)
    precision = precision_score(y_true, y_pred_binary, zero_division=0)
    recall = recall_score(y_true, y_pred_binary, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {"f1": f1, "precision": precision, "recall": recall, "fpr": fpr}


def find_best_threshold(y_true, y_probs):
    """
    FIX #4: instead of the hardcoded 0.5 cutoff, scan the precision-recall
    curve (computed on validation data) and pick the threshold that
    maximizes F1. This alone can fix a zero-recall model without touching
    the architecture, if the model's probabilities do carry real signal
    (i.e. AUC-ROC is meaningfully above 0.5).
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    # precision_recall_curve returns len(thresholds) = len(precisions) - 1
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-12)
    if len(f1s) == 0:
        return 0.5
    best_idx = np.argmax(f1s)
    return float(thresholds[best_idx])


def evaluate_baseline_vs_model(model, X_train, y_train, X_val, y_val, X_test, y_test):
    """
    FIXED (leakage): baseline fit ONLY on train, evaluated ONLY on held-out test.
    FIX #2: reports AUC-ROC for both baseline and model (threshold-independent).
    FIX #4: decision threshold for the world model is tuned on the VALIDATION
    split (never on test) via find_best_threshold(), instead of hardcoded 0.5.
    """
    # --- Logistic regression baseline ---
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
    baseline.fit(X_train_flat, y_train)
    baseline_probs = baseline.predict_proba(X_test_flat)[:, 1]
    baseline_pred = (baseline_probs >= 0.5).astype(int)
    baseline_metrics = compute_metrics(y_test, baseline_pred)
    baseline_metrics["auc_roc"] = (
        roc_auc_score(y_test, baseline_probs) if len(np.unique(y_test)) > 1 else float("nan")
    )

    # --- World model: tune threshold on VALIDATION set first ---
    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.from_numpy(X_val).to(DEVICE)
        _, val_pred_infil, _ = model(X_val_tensor)
        val_probs = val_pred_infil.cpu().numpy()
    tuned_threshold = find_best_threshold(y_val, val_probs)

    with torch.no_grad():
        X_test_tensor = torch.from_numpy(X_test).to(DEVICE)
        _, test_pred_infil, _ = model(X_test_tensor)
        model_pred_prob = test_pred_infil.cpu().numpy()
    model_pred_binary = (model_pred_prob >= tuned_threshold).astype(int)
    model_metrics = compute_metrics(y_test, model_pred_binary)
    model_metrics["auc_roc"] = (
        roc_auc_score(y_test, model_pred_prob) if len(np.unique(y_test)) > 1 else float("nan")
    )
    model_metrics["threshold_used"] = tuned_threshold

    return baseline_metrics, model_metrics


# ----------------------------------------------------------------------------
# STEP 7: Save model weights + training config
# ----------------------------------------------------------------------------
def save_artifacts(model, config, out_prefix="world_model"):
    torch.save(model.state_dict(), f"{out_prefix}.pt")

    # config may contain a torch.Tensor (pos_class_weight) -- convert to plain
    # Python types so json.dump doesn't choke on it.
    json_safe_config = {
        k: (v.item() if isinstance(v, torch.Tensor) else v)
        for k, v in config.items()
    }
    with open(f"{out_prefix}_config.json", "w") as f:
        json.dump(json_safe_config, f, indent=2)
    print(f"Saved weights -> {out_prefix}.pt")
    print(f"Saved config  -> {out_prefix}_config.json")


# ----------------------------------------------------------------------------
# MAIN — wires steps 1-7 together
# ----------------------------------------------------------------------------
def main():
    print(f"Using device: {DEVICE}\n")

    # STEP 1
    features, labels = load_feature_matrix()
    n_features = features.shape[1]

    # STEP 2
    X, next_state, next_label = make_windows(features, labels, CONFIG["window_size"])
    print(f"Total windows: {len(X)}  |  window_size={CONFIG['window_size']}  |  n_features={n_features}")

    # FIX #1: report actual positive rate in the windowed labels
    positive_rate = next_label.mean()
    print(f"Positive rate (infiltration=1) in windowed labels: "
          f"{positive_rate:.4f}  ({int(next_label.sum())} / {len(next_label)} windows)\n")

    # Chronological 3-way split: train / val / test (never shuffle time-series!)
    # val is needed so the decision threshold (Fix #4) is tuned WITHOUT
    # touching test data -- tuning on test would just be leakage again.
    n = len(X)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    X_train, X_val, X_test = X[:train_end], X[train_end:val_end], X[val_end:]
    ns_train, ns_val, ns_test = next_state[:train_end], next_state[train_end:val_end], next_state[val_end:]
    y_train, y_val, y_test = next_label[:train_end], next_label[train_end:val_end], next_label[val_end:]

    # FIX #3: class weight computed FROM TRAIN DATA ONLY, proportional to imbalance
    train_pos_rate = max(y_train.mean(), 1e-6)
    pos_weight_value = (1 - train_pos_rate) / train_pos_rate  # e.g. 9:1 imbalance -> weight ~9
    CONFIG["pos_class_weight"] = torch.tensor(pos_weight_value, device=DEVICE)
    print(f"Train positive rate: {train_pos_rate:.4f}  ->  pos_class_weight={pos_weight_value:.2f}\n")

    train_ds = WindowDataset(X_train, ns_train, y_train)
    val_ds = WindowDataset(X_val, ns_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    # STEP 3
    model = WorldModel(
        n_features=n_features,
        hidden_size=CONFIG["hidden_size"],
        num_layers=CONFIG["num_layers"],
    ).to(DEVICE)

    # STEP 4
    print("Training world model...\n")
    model = train_model(model, train_loader, val_loader, CONFIG)

    # STEP 5 — demo the K-step rollout on the very first test window
    print(f"\nRunning {CONFIG['k_steps']}-step forward rollout on a sample window...")
    probs = k_step_rollout(model, X_test[0], CONFIG["k_steps"])
    for step, p in enumerate(probs, start=1):
        print(f"  t+{step}: P(infiltration) = {p:.4f}")
    plot_rollout_curve(probs)

    # STEP 6 — required deliverable: compare against logistic regression baseline
    print("\nComparing against logistic regression baseline...")
    baseline_metrics, model_metrics = evaluate_baseline_vs_model(
        model, X_train, y_train, X_val, y_val, X_test, y_test
    )
    print(f"  Baseline  (LogReg) -> {baseline_metrics}")
    print(f"  World Model (LSTM) -> {model_metrics}")

    # Honest reporting: recall must be a REAL non-zero number, not a
    # degenerate always-negative trick, before calling this a "win"
    if model_metrics["recall"] == 0.0:
        print("\n  ⚠️  World model recall is still 0 -- it is NOT detecting any "
              "real attacks. Do NOT report 'beats baseline' as a win. "
              "Check AUC-ROC: if AUC is also ~0.5, the model hasn't learned "
              "signal yet (needs more/better data or architecture changes). "
              "If AUC is well above 0.5 but recall is 0, the threshold is "
              "still miscalibrated -- revisit find_best_threshold / val set size.")
        beats_baseline = False
    else:
        beats_baseline = (
            any(model_metrics[k] > baseline_metrics[k] for k in ["f1", "precision", "recall", "auc_roc"])
            or model_metrics["fpr"] < baseline_metrics["fpr"]
        )
    print(f"  Model beats baseline on at least one metric (recall-checked): {beats_baseline}")

    # STEP 7
    save_artifacts(model, CONFIG)


if __name__ == "__main__":
    main()

Using device: cpu

Total windows: 4989  |  window_size=10  |  n_features=16
Positive rate (infiltration=1) in windowed labels: 0.0888  (443 / 4989 windows)

Train positive rate: 0.0747  ->  pos_class_weight=12.38

Training world model...

Epoch  1/15  train_loss=2.2442  val_loss=2.6570  val_auc=0.4987  <- best so far (by AUC)
Epoch  2/15  train_loss=2.2246  val_loss=2.7395  val_auc=0.5094  <- best so far (by AUC)
Epoch  3/15  train_loss=2.2058  val_loss=2.6806  val_auc=0.5160  <- best so far (by AUC)
Epoch  4/15  train_loss=2.1913  val_loss=2.6297  val_auc=0.5225  <- best so far (by AUC)
Epoch  5/15  train_loss=2.1728  val_loss=2.6174  val_auc=0.5313  <- best so far (by AUC)
Epoch  6/15  train_loss=2.1479  val_loss=2.7251  val_auc=0.5345  <- best so far (by AUC)
Epoch  7/15  train_loss=2.1294  val_loss=2.6994  val_auc=0.5483  <- best so far (by AUC)
Epoch  8/15  train_loss=2.0886  val_loss=2.6700  val_auc=0.5794  <- best so far (by AUC)
Epoch  9/15  train_loss=2.0137  val_loss=3.1530  